# 01 — Data exploration + mini Phase-4 sanity check (PLAN §9)

Goal per PLAN §9 steps 4–6:
1. Identify one SARS-CoV-2 + one influenza A dataset on cellxgene downloadable in <1hr each.
2. Read into AnnData.
3. Compute mean(infected) - mean(mock) response vector per virus.
4. Pearson correlation between the two viruses' response vectors.
5. Gate: corr < 0.7 → signal exists, proceed to full Phase 2. corr > 0.9 → stop, reframe.

**Dataset choice.** Lee et al. 2020, *Sci Immunology* (cellxgene `de2c780c-1747-40bd-9ccf-9588ec186cee`) contains PBMC scRNA-seq from COVID-19 patients, influenza A patients, and healthy donors in a *single* harmonized study (59,572 cells). Using one study removes batch/pipeline confounds from the correlation check, and we can subset it into two virus-vs-mock AnnData objects for the §9 protocol. Faithful to PLAN intent; cleaner than stitching two unrelated studies for an hour-long sanity check.

Runs on laptop CPU. Network-bound for the initial download (~few hundred MB).

In [ ]:
import anndata as ad
import cellxgene_census
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import pearsonr, spearmanr

sc.settings.verbosity = 1
CENSUS_VERSION = "2025-11-08"  # pin for reproducibility (Census stable as of writing)
DATASET_ID = "de2c780c-1747-40bd-9ccf-9588ec186cee"  # Lee et al. 2020

## Download — stream the Lee et al. dataset from Census
Pulls a single AnnData with all cells; we'll subset by `disease` next.

In [ ]:
with cellxgene_census.open_soma(census_version=CENSUS_VERSION) as census:
    adata = cellxgene_census.get_anndata(
        census=census,
        organism="Homo sapiens",
        obs_value_filter=f"dataset_id == '{DATASET_ID}'",
    )
print(adata)
print("obs columns:", list(adata.obs.columns))

In [ ]:
# Inspect disease labels to pick mock / SARS-CoV-2 / IAV cohorts
print(adata.obs["disease"].value_counts())
print()
print(adata.obs[["disease", "cell_type"]].value_counts().head(20))

## Subset into two virus-vs-mock AnnData objects
Census `disease` ontology terms used: `COVID-19`, `influenza`, `normal`.

In [ ]:
# Resolve disease labels case-insensitively — cellxgene uses ontology strings
diseases = adata.obs["disease"].astype(str)
covid_mask = diseases.str.contains("COVID-19", case=False, na=False)
iav_mask = diseases.str.contains("influenza", case=False, na=False)
mock_mask = diseases.str.lower().eq("normal")

print(f"COVID-19 cells:  {covid_mask.sum():>7}")
print(f"Influenza cells: {iav_mask.sum():>7}")
print(f"Healthy cells:   {mock_mask.sum():>7}")

adata_covid = adata[covid_mask | mock_mask].copy()
adata_covid.obs["infection_status"] = np.where(
    adata_covid.obs["disease"].astype(str).str.contains("COVID-19", case=False), "infected", "mock"
)

adata_iav = adata[iav_mask | mock_mask].copy()
adata_iav.obs["infection_status"] = np.where(
    adata_iav.obs["disease"].astype(str).str.contains("influenza", case=False), "infected", "mock"
)

print(adata_covid)
print(adata_iav)

## Light QC + normalization
Just enough so means are comparable. Full QC pipeline lives in Phase 3.

In [ ]:
def quick_norm(a: ad.AnnData) -> ad.AnnData:
    a = a.copy()
    # cellxgene matrix is raw counts; normalize per cell, log1p
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    return a


adata_covid = quick_norm(adata_covid)
adata_iav = quick_norm(adata_iav)
print("Normalized.")

## Compute response vectors per virus
Response = mean(infected) - mean(mock) across all genes, on the shared gene set.

In [ ]:
def response_vector(a: ad.AnnData) -> pd.Series:
    inf = a[a.obs["infection_status"] == "infected"].X
    mock = a[a.obs["infection_status"] == "mock"].X
    inf_mean = np.asarray(inf.mean(axis=0)).ravel()
    mock_mean = np.asarray(mock.mean(axis=0)).ravel()
    return pd.Series(inf_mean - mock_mean, index=a.var_names)


rv_covid = response_vector(adata_covid)
rv_iav = response_vector(adata_iav)

shared = rv_covid.index.intersection(rv_iav.index)
rv_covid = rv_covid.loc[shared]
rv_iav = rv_iav.loc[shared]
print(f"Shared genes: {len(shared)}")
print("Top 10 COVID up-regulated:", rv_covid.nlargest(10).index.tolist())
print("Top 10 IAV   up-regulated:", rv_iav.nlargest(10).index.tolist())

## Gate decision (PLAN §9 step 6)

In [ ]:
pearson_r, pearson_p = pearsonr(rv_covid.values, rv_iav.values)
spearman_r, spearman_p = spearmanr(rv_covid.values, rv_iav.values)

print(f"Pearson  r = {pearson_r:.4f}  (p = {pearson_p:.2e})")
print(f"Spearman r = {spearman_r:.4f}  (p = {spearman_p:.2e})")
print()

if pearson_r > 0.9:
    verdict = "STOP — response vectors too correlated. Insufficient virus-specific signal. Reframe project."
elif pearson_r < 0.7:
    verdict = "PROCEED — viruses are genuinely distinct. Continue to full Phase 2 data acquisition."
else:
    verdict = "GREY ZONE (0.7 ≤ r ≤ 0.9) — marginal. Proceed but expect transfer learning to be subtle. Add more datasets in Phase 2 before re-running gate."

print("GATE VERDICT:", verdict)

## Top differentially expressed genes per virus (sanity peek at ISG overlap)
Not a formal Phase-4 ISG check (that's `02_sanity_check_signal.ipynb`); just an eyeball.

In [ ]:
top_covid = set(rv_covid.nlargest(100).index)
top_iav = set(rv_iav.nlargest(100).index)
jaccard = len(top_covid & top_iav) / len(top_covid | top_iav)
print(f"Top-100 up-regulated gene Jaccard (COVID ∩ IAV): {jaccard:.3f}")
print(f"Shared in top-100: {sorted(top_covid & top_iav)[:30]}")